<a href="https://colab.research.google.com/github/Damainx22/RGV-Business-Survival/blob/main/notebooks/02_acs_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# Notebook 02: ACS Demographics Data Pull
# Purpose: Pull ACS 5-year estimates for RGV
#          zip codes via Census API for 2018-2022,
#          clean and save to data/cleaned/
# ============================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Standard imports
import os
import pandas as pd
import requests

# Define folder paths
RAW = '/content/drive/MyDrive/rgv_business_survival/data/raw'
CLEANED = '/content/drive/MyDrive/rgv_business_survival/data/cleaned'
MERGED = '/content/drive/MyDrive/rgv_business_survival/data/merged'

# API key
API_KEY = "API KEY"

print("Setup complete")

Mounted at /content/drive
Setup complete


In [3]:
def pull_acs_year(year, api_key):
    # Variables we want from Census ACS:
    variables = "NAME,B19013_001E,B01003_001E,B17001_002E,B23025_005E,B23025_002E"

    # Census changed their API in 2020 — can no longer filter by state for ZCTAs
    # Pre-2020: filter to Texas (state code 48) to speed up the request
    # 2020+: pull all US ZCTAs and filter to RGV after
    if year >= 2020:
        url = f"https://api.census.gov/data/{year}/acs/acs5?get={variables}&for=zip%20code%20tabulation%20area:*&key={api_key}"
    else:
        url = f"https://api.census.gov/data/{year}/acs/acs5?get={variables}&for=zip%20code%20tabulation%20area:*&in=state:48&key={api_key}"

    # Make the HTTP request to Census API and parse JSON response
    # data[0] = header row (column names), data[1:] = actual data rows
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # Rename cryptic Census variable codes to readable column names
    df = df.rename(columns={
        'B19013_001E': 'median_income',
        'B01003_001E': 'total_population',
        'B17001_002E': 'poverty_count',
        'B23025_005E': 'unemployed_count',
        'B23025_002E': 'labor_force_count',
        'zip code tabulation area': 'zip_code',
    })

    # Convert demographic columns from strings to numbers
    # errors='coerce' turns any bad values into NaN instead of crashing
    numeric_cols = ['median_income', 'total_population', 'poverty_count',
                    'unemployed_count', 'labor_force_count']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Calculate rates instead of raw counts
    # Rates are more useful for the model since they normalize across zip codes of different sizes
    # A zip code with 500 poor people out of 50,000 is very different from 500 out of 1,000
    df['poverty_rate'] = df['poverty_count'] / df['total_population']
    df['unemployment_rate'] = df['unemployed_count'] / df['labor_force_count']

    # Tag each row with the year it came from
    # Needed later when we merge with SBA loan data by zip code AND year
    df['year'] = year

    # Filter down to only RGV zip codes (all start with 785)
    # Covers Hidalgo, Cameron, Starr, and Willacy counties
    rgv = df[df['zip_code'].str.startswith('785')]

    return rgv

In [5]:
# Loop through each year and call the pull function
# Results from each year get appended to all_years list
all_years = []
for year in [2018, 2019, 2020, 2021, 2022]:
    result = pull_acs_year(year, API_KEY)
    all_years.append(result)
    print(f'  Pulled {year}: {len(result)} zip codes')

# Stack all 5 years into one DataFrame
# ignore_index=True resets row numbers so they don't repeat across years
acs_combined = pd.concat(all_years, ignore_index=True)

# Confirm total rows — should be ~59 zip codes × 5 years = ~295 rows
print(f'Total rows: {len(acs_combined)}')
print(f'Columns: {list(acs_combined.columns)}')

# Save combined ACS data to cleaned folder
# This file will be merged with SBA loan data in notebook 05
acs_combined.to_csv(f'{CLEANED}/acs_rgv_clean.csv', index=False)
print(f'Saved: {len(acs_combined)} rows to acs_rgv_clean.csv')

  Pulled 2018: 58 zip codes
  Pulled 2019: 58 zip codes
  Pulled 2020: 58 zip codes
  Pulled 2021: 61 zip codes
  Pulled 2022: 61 zip codes
Total rows: 296
Columns: ['NAME', 'median_income', 'total_population', 'poverty_count', 'unemployed_count', 'labor_force_count', 'state', 'zip_code', 'poverty_rate', 'unemployment_rate', 'year']
Saved: 296 rows to acs_rgv_clean.csv
